### Stage-wise spend — Month on Month

Spend from `leads_analytics.masterspend` → funnel via `cid → leads.utm_campaign → leadstagehistories`.

**MoM attribution (same calendar month):**
- Spend → `dateist` month
- Lead count → leads **born** that month (CID had spend that month)
- Site visit / Closure count → first **Post Site Visit** / **Sales Closure** in that month

| Column | Meaning |
|--------|--------|
| Month | `YYYY-MM` |
| Spend | Total media spend (₹) |
| Lead count / Spend per lead | Distinct born leads / Spend ÷ that |
| Site visit count / Spend per site visit | Distinct PSV leads / Spend ÷ that |
| Closure count / Spend per closure | Distinct SC leads / Spend ÷ that |

Filters: `START_DATE`, `END_DATE`, `PROJECTS`, `PLATFORMS` → Run All → Excel.


In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import q, write_excel

# --- filters ---
START_DATE = date(2026, 1, 1)
END_DATE = date.today()  # inclusive
PROJECTS: tuple[str, ...] = ()   # e.g. ("BROADWAY", "LOFT")
PLATFORMS: tuple[str, ...] = ()  # e.g. ("Google", "FIM")

START_TS = f"TIMESTAMP '{START_DATE.isoformat()} 00:00:00'"
END_TS = f"TIMESTAMP '{(END_DATE + timedelta(days=1)).isoformat()} 00:00:00'"
START_D, END_D = START_DATE.isoformat(), END_DATE.isoformat()


def _in_list(col: str, values: tuple[str, ...]) -> str:
    if not values:
        return "TRUE"
    quoted = ", ".join("'" + v.replace("'", "''") + "'" for v in values)
    return f"TRIM(CAST({col} AS varchar)) IN ({quoted})"


proj_f = _in_list("projectname", PROJECTS)
plat_f = _in_list("platform", PLATFORMS)
print("Window", START_D, "→", END_D, "| Projects", PROJECTS or "(all)", "| Platforms", PLATFORMS or "(all)")


In [ ]:
# Month × Platform × Project
SQL_MOM = f"""
WITH spend AS (
  SELECT
    date_format(CAST(dateist AS date), '%Y-%m') AS month_key,
    date_format(CAST(dateist AS date), '%M %Y') AS month_name,
    TRIM(CAST(cid AS varchar)) AS cid,
    COALESCE(NULLIF(TRIM(CAST(platform AS varchar)), ''), '(blank)') AS platform,
    COALESCE(NULLIF(TRIM(CAST(projectname AS varchar)), ''), '(blank)') AS project,
    SUM(TRY_CAST(amountspent AS double)) AS spend
  FROM leads_analytics.masterspend
  WHERE CAST(dateist AS date) >= DATE '{START_D}'
    AND CAST(dateist AS date) <= DATE '{END_D}'
    AND TRIM(CAST(cid AS varchar)) IS NOT NULL
    AND TRIM(CAST(cid AS varchar)) <> ''
    AND TRY_CAST(amountspent AS double) > 0
    AND {proj_f}
    AND {plat_f}
  GROUP BY 1, 2, 3, 4, 5
),
leads_cid AS (
  SELECT
    date_format(CAST(l.createdat AS date), '%Y-%m') AS month_key,
    TRIM(CAST(l.utm_campaign AS varchar)) AS cid,
    CAST(l._id AS varchar) AS lead_oid,
    COALESCE(NULLIF(TRIM(CAST(p.name AS varchar)), ''), '(blank)') AS project
  FROM "leadsdb-prod".leads l
  LEFT JOIN "leadsdb-prod".projects p
    ON CAST(p._id AS varchar) = CAST(l.project AS varchar)
  WHERE l.utm_campaign IS NOT NULL
    AND TRIM(CAST(l.utm_campaign AS varchar)) <> ''
    AND l.createdat >= {START_TS} AND l.createdat < {END_TS}
),
first_stage AS (
  SELECT
    CAST(lead AS varchar) AS lead_oid,
    CAST(stage AS varchar) AS stage,
    date_format(CAST(MIN(createdat) AS date), '%Y-%m') AS month_key
  FROM "leadsdb-prod".leadstagehistories
  WHERE CAST(stage AS varchar) IN ('Post Site Visit', 'Sales Closure')
    AND createdat >= {START_TS} AND createdat < {END_TS}
  GROUP BY 1, 2
),
plat_spend AS (
  SELECT month_key, month_name, platform, project,
         SUM(spend) AS spend, COUNT(DISTINCT cid) AS cid_count
  FROM spend
  GROUP BY 1, 2, 3, 4
),
plat_funnel AS (
  SELECT
    sp.month_key,
    sp.platform,
    sp.project,
    COUNT(DISTINCT lf.lead_oid) AS lead_count,
    COUNT(DISTINCT CASE WHEN psv.lead_oid IS NOT NULL THEN psv.lead_oid END) AS site_visit_count,
    COUNT(DISTINCT CASE WHEN sc.lead_oid IS NOT NULL THEN sc.lead_oid END) AS closure_count
  FROM (SELECT DISTINCT month_key, platform, project, cid FROM spend) sp
  LEFT JOIN leads_cid lf
    ON lf.cid = sp.cid AND lf.project = sp.project AND lf.month_key = sp.month_key
  LEFT JOIN first_stage psv
    ON psv.lead_oid = lf.lead_oid AND psv.stage = 'Post Site Visit' AND psv.month_key = sp.month_key
  LEFT JOIN first_stage sc
    ON sc.lead_oid = lf.lead_oid AND sc.stage = 'Sales Closure' AND sc.month_key = sp.month_key
  GROUP BY 1, 2, 3
)
SELECT
  ps.month_key,
  ps.month_name,
  ps.platform,
  ps.project,
  ROUND(ps.spend, 2) AS spend,
  ps.cid_count,
  COALESCE(pf.lead_count, 0) AS lead_count,
  ROUND(ps.spend / NULLIF(pf.lead_count, 0), 2) AS spend_per_lead,
  COALESCE(pf.site_visit_count, 0) AS site_visit_count,
  ROUND(ps.spend / NULLIF(pf.site_visit_count, 0), 2) AS spend_per_site_visit,
  COALESCE(pf.closure_count, 0) AS closure_count,
  ROUND(ps.spend / NULLIF(pf.closure_count, 0), 2) AS spend_per_closure
FROM plat_spend ps
LEFT JOIN plat_funnel pf
  ON pf.month_key = ps.month_key
 AND pf.platform = ps.platform
 AND pf.project = ps.project
ORDER BY ps.month_key, ps.spend DESC
"""

raw = q(SQL_MOM, database="leads_analytics")
RENAME = {
    "month_key": "Month key",
    "month_name": "Month",
    "platform": "Platform",
    "project": "Project",
    "spend": "Spend",
    "cid_count": "CID count",
    "lead_count": "Lead count",
    "spend_per_lead": "Spend per lead",
    "site_visit_count": "Site visit count",
    "spend_per_site_visit": "Spend per site visit",
    "closure_count": "Closure count",
    "spend_per_closure": "Spend per closure",
}
by_month_plat_proj = raw.rename(columns=RENAME)
display(by_month_plat_proj.head(20))
print("rows", len(by_month_plat_proj), "spend", float(by_month_plat_proj["Spend"].sum()))


In [ ]:
# MoM × Platform (projects rolled up)
by_month_plat = (
    by_month_plat_proj.groupby(["Month key", "Month", "Platform"], as_index=False)
    .agg(
        **{
            "Spend": ("Spend", "sum"),
            "CID count": ("CID count", "sum"),
            "Lead count": ("Lead count", "sum"),
            "Site visit count": ("Site visit count", "sum"),
            "Closure count": ("Closure count", "sum"),
        }
    )
    .sort_values(["Month key", "Spend"], ascending=[True, False])
)
by_month_plat["Spend per lead"] = (by_month_plat["Spend"] / by_month_plat["Lead count"].replace(0, pd.NA)).round(2)
by_month_plat["Spend per site visit"] = (
    by_month_plat["Spend"] / by_month_plat["Site visit count"].replace(0, pd.NA)
).round(2)
by_month_plat["Spend per closure"] = (
    by_month_plat["Spend"] / by_month_plat["Closure count"].replace(0, pd.NA)
).round(2)
by_month_plat = by_month_plat[
    [
        "Month", "Month key", "Platform", "Spend", "CID count",
        "Lead count", "Spend per lead",
        "Site visit count", "Spend per site visit",
        "Closure count", "Spend per closure",
    ]
]

# MoM overall (unique leads per month across platforms)
SQL_MOM_ALL = f"""
WITH spend_cid AS (
  SELECT
    date_format(CAST(dateist AS date), '%Y-%m') AS month_key,
    date_format(CAST(dateist AS date), '%M %Y') AS month_name,
    TRIM(CAST(cid AS varchar)) AS cid,
    SUM(TRY_CAST(amountspent AS double)) AS spend
  FROM leads_analytics.masterspend
  WHERE CAST(dateist AS date) >= DATE '{START_D}'
    AND CAST(dateist AS date) <= DATE '{END_D}'
    AND TRIM(CAST(cid AS varchar)) IS NOT NULL
    AND TRIM(CAST(cid AS varchar)) <> ''
    AND TRY_CAST(amountspent AS double) > 0
    AND {proj_f}
    AND {plat_f}
  GROUP BY 1, 2, 3
),
spend_m AS (
  SELECT month_key, month_name, SUM(spend) AS spend, COUNT(DISTINCT cid) AS cid_count
  FROM spend_cid GROUP BY 1, 2
),
leads_cid AS (
  SELECT
    date_format(CAST(createdat AS date), '%Y-%m') AS month_key,
    TRIM(CAST(utm_campaign AS varchar)) AS cid,
    CAST(_id AS varchar) AS lead_oid
  FROM "leadsdb-prod".leads
  WHERE utm_campaign IS NOT NULL
    AND TRIM(CAST(utm_campaign AS varchar)) <> ''
    AND createdat >= {START_TS} AND createdat < {END_TS}
),
first_stage AS (
  SELECT
    CAST(lead AS varchar) AS lead_oid,
    CAST(stage AS varchar) AS stage,
    date_format(CAST(MIN(createdat) AS date), '%Y-%m') AS month_key
  FROM "leadsdb-prod".leadstagehistories
  WHERE CAST(stage AS varchar) IN ('Post Site Visit', 'Sales Closure')
    AND createdat >= {START_TS} AND createdat < {END_TS}
  GROUP BY 1, 2
),
funnel AS (
  SELECT
    sc.month_key,
    COUNT(DISTINCT l.lead_oid) AS lead_count,
    COUNT(DISTINCT CASE WHEN psv.lead_oid IS NOT NULL THEN psv.lead_oid END) AS site_visit_count,
    COUNT(DISTINCT CASE WHEN scl.lead_oid IS NOT NULL THEN scl.lead_oid END) AS closure_count
  FROM (SELECT DISTINCT month_key, cid FROM spend_cid) sc
  LEFT JOIN leads_cid l ON l.cid = sc.cid AND l.month_key = sc.month_key
  LEFT JOIN first_stage psv
    ON psv.lead_oid = l.lead_oid AND psv.stage = 'Post Site Visit' AND psv.month_key = sc.month_key
  LEFT JOIN first_stage scl
    ON scl.lead_oid = l.lead_oid AND scl.stage = 'Sales Closure' AND scl.month_key = sc.month_key
  GROUP BY 1
)
SELECT
  sm.month_key,
  sm.month_name,
  ROUND(sm.spend, 2) AS spend,
  sm.cid_count,
  COALESCE(f.lead_count, 0) AS lead_count,
  ROUND(sm.spend / NULLIF(f.lead_count, 0), 2) AS spend_per_lead,
  COALESCE(f.site_visit_count, 0) AS site_visit_count,
  ROUND(sm.spend / NULLIF(f.site_visit_count, 0), 2) AS spend_per_site_visit,
  COALESCE(f.closure_count, 0) AS closure_count,
  ROUND(sm.spend / NULLIF(f.closure_count, 0), 2) AS spend_per_closure
FROM spend_m sm
LEFT JOIN funnel f ON f.month_key = sm.month_key
ORDER BY sm.month_key
"""
mom_all = q(SQL_MOM_ALL, database="leads_analytics").rename(columns={
    "month_key": "Month key", "month_name": "Month", "spend": "Spend", "cid_count": "CID count",
    "lead_count": "Lead count", "spend_per_lead": "Spend per lead",
    "site_visit_count": "Site visit count", "spend_per_site_visit": "Spend per site visit",
    "closure_count": "Closure count", "spend_per_closure": "Spend per closure",
})
mom_all = mom_all[
    ["Month", "Month key", "Spend", "CID count",
     "Lead count", "Spend per lead",
     "Site visit count", "Spend per site visit",
     "Closure count", "Spend per closure"]
]

filters_df = pd.DataFrame([
    {"key": "start_date", "value": START_D},
    {"key": "end_date", "value": END_D},
    {"key": "projects", "value": ", ".join(PROJECTS) if PROJECTS else "(all)"},
    {"key": "platforms", "value": ", ".join(PLATFORMS) if PLATFORMS else "(all)"},
    {"key": "grain", "value": "month × platform [× project]"},
    {"key": "attribution", "value": "same calendar month for spend, born leads, first PSV, first SC"},
])

write_excel(
    ROOT / "StageWise_Spend_Analysis.xlsx",
    {
        "MoM_Overall": mom_all,
        "MoM_By_Platform": by_month_plat,
        "MoM_By_Platform_Project": by_month_plat_proj,
        "Filters": filters_df,
    },
)
display(mom_all)
display(by_month_plat.head(20))
